[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/08_Agua_Superficial/01_agua_superficial.ipynb)

# Agua superficial: ocurrencia, estacionalidad, recurrencia y cambio (JRC)
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [1]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

Librerías listas.
earthengine-api: 1.7.39
geemap: 0.38.8


In [2]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "gen-lang-client-0636031685"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

Earth Engine inicializado correctamente.
Proyecto: gen-lang-client-0636031685


## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [3]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

Zona: Escoma, La Paz (prueba)
Ventana analítica: 1996-01-01 -> 2026-12-31 (31 años calendario)
Años completos para métricas: 1996 - 2025
Calendario: siembra mes 11 | cosecha mes 4
Umbrales activos: {'helada_c': 0.0, 'helada_grave_c': -2.0, 'mes_seco_mm': 30.0, 'spi_severo': -1.5, 'ndvi_anomalia_z': -1.5, 'manantial_occurrencia': 100, 'manantial_drenaje_max': 50, 'bofedal_altitud_min': 3000, 'bofedal_altitud_max': 5200, 'bofedal_pendiente_max': 5.0, 'bofedal_seasonality': 8, 'bofedal_ndvi_seco': 0.25, 'clases_agricolas': [12, 14]}


In [4]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

Utilidades: apilar(), reducir_bandas(), composite_mensual(), promediar_aoi(), serie_aoi()


In [5]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

Mounted at /content/drive
Descargando AOI desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N
To: /content/zona_aoi.geojson
100%|██████████| 6.32k/6.32k [00:00<00:00, 10.6MB/s]


AOI cargado desde: /content/zona_aoi.geojson
Zona: Escoma, La Paz (prueba)
Área del AOI: 5,638.9 km²
Extensión (bbox): [-68.088142, -19.88671, -67.170426, -19.050138]
Puntos de saber local cargados: {'manantiales': 0, 'bofedales': 0}


Map(center=[-19.436538457151876, -67.61413826547425], controls=(WidgetControl(options=['position', 'transparen…

# M8. Agua superficial: ocurrencia, estacionalidad, recurrencia y cambio (JRC)
**Dimensiones:** *ocurrencia* (% del tiempo con agua), *estacionalidad* (nº de meses con agua),
*recurrencia* (% de meses húmedos que repiten agua), *tendencia/cambio* entre épocas.

* **Primario:** `JRC/GSW1_4/GlobalSurfaceWater` + `YearlyHistory` — 30 m, **1984–2021**
  (período nativo fijo del producto: no se puede extender con este dataset; se declara y se
  contrasta con datos recientes si hace falta).
* **Bandas usadas:** `occurrence` (0–100% = **ocurrencia**, no "frecuencia"), `seasonality`
  (0–12 meses), `recurrence` (0–100%), `change_abs` (cambio de ocurrencia entre épocas),
  `YearlyHistory.waterClass` (2 = agua estacional, 3 = agua permanente, por año).
* **Alternativas:** Sentinel-2 MNDWI (para agua post-2021) o `GOOGLE/DYNAMICWORLD`
  (10 m, 2015+, probabilidad de agua) — comentadas en M8.1.

<details>
<summary><b>📐 Corrección de uso (viene del MD guía)</b> — clic para ver</summary>
<br>

En JRC, `occurrence` = **ocurrencia** = % del tiempo que el píxel tuvo agua *en el período
completo* (1984–2021). La **frecuencia** en esta plantilla siempre es un conteo de eventos
(p. ej. nº de años con agua), y la **recurrencia** es % de meses húmedos (jun–sep) en los que
hubo agua. No confundir los tres.

</details>

In [8]:
# ============================================================
# M8.3 + M8.4 Frecuencia vs Recurrencia + red de drenaje
# ------------------------------------------------------------
# Frecuencia  = % de años con agua (1984-2021)   → estabilidad
# Recurrencia = % de meses húmedos con agua      → confiabilidad
# Drenaje     = cauces derivados de MERIT Hydro (upa > umbral)
# Todos recortados al GeoJSON (roi) y en HBox 50/50.
# ============================================================
from ipywidgets import HBox, Layout

# ------------------------------------------------------------
# 0) DRENAJE — red hídrica desde MERIT Hydro
# ------------------------------------------------------------
# Banda "upa" = área acumulada aguas arriba (km²).
# Umbral bajo (5–10 km²) → red densa con quebradas.
# Umbral alto (25–50 km²) → solo ríos principales.
DRENAJE_UPA_KM2 = 10

drenaje = (
    ee.Image("MERIT/Hydro/v1_0_1")
      .select("upa")
      .gt(DRENAJE_UPA_KM2)      # solo donde acumula > umbral aguas arriba
      .selfMask()                # fuera del cauce queda transparente
      .clip(roi)
      .rename("drenaje")
)
vis_drenaje = {"palette": ["00a000"]}   # verde

print(f"Drenaje derivado: upa > {DRENAJE_UPA_KM2} km² (MERIT Hydro)")

# ------------------------------------------------------------
# 1) FRECUENCIA — calculada desde YearlyHistory
# ------------------------------------------------------------
yh_col = ee.ImageCollection("JRC/GSW1_4/YearlyHistory").filterBounds(roi)

freq_anios = (yh_col.map(lambda img: img.select("waterClass").gte(2)
                                                .unmask(0).rename("w"))
                    .sum().rename("frecuencia"))

n_anios_obs = (yh_col.map(lambda img: img.select("waterClass").gt(0)
                                                .unmask(0).rename("n"))
                     .sum().rename("n_obs"))

freq_pct = (freq_anios.divide(n_anios_obs).multiply(100)
            .rename("frecuencia_pct")
            .updateMask(n_anios_obs.gt(0))
            .clip(roi))

stats_freq = freq_pct.reduceRegion(
    ee.Reducer.percentile([10, 50, 90]).combine(
        ee.Reducer.mean(), sharedInputs=True),
    roi, ESCALAS["jrc"], maxPixels=1e9, tileScale=8, bestEffort=True,
).getInfo()
print("\nFRECUENCIA de agua (% de años con agua, dentro del AOI):")
print(f"  media : {stats_freq['frecuencia_pct_mean']:.1f} %")
print(f"  p10   : {stats_freq['frecuencia_pct_p10']:.1f} %   (sitios más secos)")
print(f"  p90   : {stats_freq['frecuencia_pct_p90']:.1f} %   (sitios más húmedos)")

# ------------------------------------------------------------
# 2) RECURRENCIA — banda nativa de JRC GlobalSurfaceWater
# ------------------------------------------------------------
# Definir el dataset JRC principal para poder extraer la recurrencia
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

recurrencia = (jrc.select("recurrence")
                  .clip(roi))

rec_media = promediar_aoi(recurrencia, "recurrence", ESCALAS["jrc"])
print(f"\nRECURRENCIA media en el AOI: {rec_media:.1f} %")
print("  (≤ 30 → el agua superficial es poco fiable en época de lluvias)")

# ------------------------------------------------------------
# 3) Paletas
# ------------------------------------------------------------
vis_freq = {
    "min": 0, "max": 100,
    "palette": ["ffffff", "eff3ff", "c6dbef", "9ecae1",
                "6baed6", "4292c6", "2171b5", "084594", "08306b"],
}
vis_rec = {
    "min": 0, "max": 100,
    "palette": ["ffffe0", "fee391", "fec44f", "fe9929",
                "ec7014", "cc4c02", "8c2d04"],
}

# Layout 50% para cada mapa
layout_50 = Layout(width="50%", height="600px")

# ------------------------------------------------------------
# 4) Mapa izquierdo — FRECUENCIA + DRENAJE
# ------------------------------------------------------------
Map_freq = geemap.Map(basemap="HYBRID", layout=layout_50)
Map_freq.centerObject(roi, ZONA["zoom"])
Map_freq.addLayer(freq_pct, vis_freq, "Frecuencia de agua (% de años)")
Map_freq.addLayer(drenaje, vis_drenaje, f"Drenaje (upa > {DRENAJE_UPA_KM2} km²)")
Map_freq.addLayer(borde_aoi(color="FF0000", ancho=2), {}, "AOI")
Map_freq.add_colorbar(vis_freq,
                      label="Frecuencia %",
                      orientation="horizontal",
                      position="bottomright",
                      layer_name="Frecuencia (%)")

# ------------------------------------------------------------
# 5) Mapa derecho — RECURRENCIA + DRENAJE
# ------------------------------------------------------------
Map_rec = geemap.Map(basemap="HYBRID", layout=layout_50)
Map_rec.centerObject(roi, ZONA["zoom"])
Map_rec.addLayer(recurrencia, vis_rec, "Recurrencia (% de meses húmedos)")
Map_rec.addLayer(drenaje, vis_drenaje, f"Drenaje (upa > {DRENAJE_UPA_KM2} km²)")
Map_rec.addLayer(borde_aoi(color="FF0000", ancho=2), {}, "AOI")
Map_rec.add_colorbar(vis_rec,
                     label="Recurrencia (%)",
                     orientation="horizontal",
                     position="bottomright",
                     layer_name="Recurrencia (%)")

# ------------------------------------------------------------
# 6) Mostrar lado a lado
# ------------------------------------------------------------
HBox([Map_freq, Map_rec],
     layout=Layout(width="100%", justify_content="space-between"))

Drenaje derivado: upa > 10 km² (MERIT Hydro)

FRECUENCIA de agua (% de años con agua, dentro del AOI):
  media : 33.0 %
  p10   : 10.8 %   (sitios más secos)
  p90   : 69.7 %   (sitios más húmedos)

RECURRENCIA media en el AOI: 45.0 %
  (≤ 30 → el agua superficial es poco fiable en época de lluvias)


## Interpretación de M8
* **Ocurrencia alta + estacionalidad de 12 meses** = agua permanente (lagos, ríos de régimen
  constante, bofedales con agua); **ocurrencia media + estacionalidad 4–8** = lagunas estacionales
  que aparecen solo en lluvias — recurso clave en temporada seca.
* **Recurrencia** responde *qué tan seguido, entre los meses húmedos, hay agua*: si en jun–sep
  llueve y aun así la recurrencia es baja, el agua superficial es poco confiable.
* **Cambio (zona azul):** ganó agua entre épocas (posible aumento de lagunas o distorsión por
  climáticos húmedos); **zona roja:** perdió agua (secado, extracción, cambio de uso) — priorizar
  verificación en campo.
* **Serie anual:** la pendiente por año resume si el agua superficial crece o se reduce; los años
  de quiebre se comparan con sequías (M5) y con el relato local.
* El producto termina en **2021**; para agua post-2021 usar Sentinel-2/Dynamic World (M8.1).

**Preguntas guía**
1. ¿Las lagunas que ustedes conocen aparecen con estacionalidad alta o baja?
2. ¿Dónde dice el mapa que se perdió agua? ¿coincide con lo observado en campo?
3. ¿Los años más bajos de la serie de área con agua son los mismos de sequía severa (M5)?

## Complemento avanzado: extender el agua mas alla de 2021 con Sentinel-2 MNDWI

El producto JRC termina en **diciembre 2021**. Para agua **reciente** se usa
Sentinel-2 con el indice **MNDWI = (Verde − SWIR1) / (Verde + SWIR1)**.
Metodo heredado del ejercicio avanzado *Poopo / JRC + Sentinel-2* (notebook
`01_long_term_surface_water_jrc.ipynb` del repositorio Transcend): mascara de
nubes con banda `SCL`, compuestos mensuales de mediana, umbral `MNDWI > 0` = agua,
y filtro de cobertura valida ≥ 80 % (el mismo criterio de calidad del JRC mensual).
A diferencia del ejemplo del Poopo (dominio historico del lago), aqui el calculo
corre sobre **`roi` = tu GeoJSON de Bolivia** (el GeoJSON siempre manda).


In [ ]:
# ============================================================
# COMPLEMENTO: serie mensual de agua con Sentinel-2 MNDWI (2019 -> hoy)
# Adaptado del ejercicio Poopo (JRC + S2) al ROI del GeoJSON de Bolivia.
# El GeoJSON (roi) siempre manda: todo se recorta y reduce a `roi`.
# ============================================================
import datetime

S2_START = "2019-01-01"
S2_END = datetime.date.today().replace(day=1).isoformat()   # ultimo mes completo
MIN_VALID_COVERAGE = 80    # % de pixeles con dato para aceptar el mes
S2_SCALE_M = 20            # B11 (SWIR1) resuelve a 20 m -> escala de reduccion
S2_COL = "COPERNICUS/S2_SR_HARMONIZED"
# S2_COL = "COPERNICUS/S2_HARMONIZED"  # TOA: mas rapido, menos preciso

# --- Area TOTAL del AOI (denominador REAL de la cobertura) ------------------
# Se calcula UNA sola vez con maxError=1 (exacto) y NO depende de la mascara S2.
AOI_KM2 = roi.area(maxError=1).divide(1e6).getInfo()
print(f"AOI total: {AOI_KM2:,.1f} km2")


def mask_s2(image):
    """Enmascara nubes/sombras/nieve con SCL y deja B3+B11 en reflectancia."""
    scl = image.select("SCL")
    clear = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    optical = image.select(["B3", "B11"]).multiply(0.0001).updateMask(clear)
    return optical.copyProperties(image, ["system:time_start"])


s2 = (ee.ImageCollection(S2_COL).filterBounds(roi)
      .filterDate(S2_START, S2_END)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
      .map(mask_s2))
print("Periodo Sentinel-2:", S2_START, "a", S2_END)
print("Imagenes S2:", s2.size().getInfo())

# --- Iterar por meses (server-side) -----------------------------------------
n_months = ee.Date(S2_END).difference(ee.Date(S2_START), "month").round()
offsets = ee.List.sequence(0, n_months.subtract(1))


def monthly_mndwi(offset):
    """Compuesto mensual MNDWI. Meses sin imagenes devuelven banda vacia (no rompe)."""
    start = ee.Date(S2_START).advance(offset, "month")
    sub = s2.filterDate(start, start.advance(1, "month"))
    comp = ee.Image(ee.Algorithms.If(
        sub.size().gt(0),
        sub.median(),
        ee.Image.constant([0, 0]).rename(["B3", "B11"]).selfMask(),  # vacia
    ))
    mndwi = comp.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    return mndwi.set("date", start.format("YYYY-MM-dd"))


mndwi_monthly = ee.ImageCollection(offsets.map(monthly_mndwi))
print("Meses con compuesto S2:", mndwi_monthly.size().getInfo())


def _area_mes(img):
    """Agua + pixeles validos en UNA sola reduceRegion por mes.

    - agua_km2   : suma de areas de pixeles con MNDWI > 0
    - valida_km2 : suma de areas de pixeles CON DATO (denominador de cobertura)
    Ambas a la MISMA escala (S2_SCALE_M) para que sean comparables.
    """
    m = img.select("MNDWI")
    pix = ee.Image.pixelArea().divide(1e6)          # km2 por pixel (sin enmascarar)
    valida = m.mask().unmask(0, False)              # 1 donde hay dato, 0 si no
    agua = m.gt(0).unmask(0, False)                 # 1 donde MNDWI>0 (agua)
    stacked = (
        pix.updateMask(valida).rename("valida_km2")
        .addBands(pix.updateMask(agua).rename("agua_km2"))
    )
    st = stacked.reduceRegion(
        ee.Reducer.sum(), roi, S2_SCALE_M,
        maxPixels=1e10, tileScale=8,
        bestEffort=False,          # <-- NO usar bestEffort en sumas de area
    )
    return ee.Feature(None, {
        "date": img.get("date"),
        "agua_km2": st.get("agua_km2"),
        "valida_km2": st.get("valida_km2"),
    })


fc = ee.FeatureCollection(mndwi_monthly.map(_area_mes)).getInfo()
df_s2 = pd.DataFrame([f["properties"] for f in fc["features"]])
df_s2["date"] = pd.to_datetime(df_s2["date"])
df_s2["agua_km2"]   = pd.to_numeric(df_s2["agua_km2"],   errors="coerce")
df_s2["valida_km2"] = pd.to_numeric(df_s2["valida_km2"], errors="coerce")

# Cobertura REAL = pixeles con dato / area total del AOI (AOI_KM2, constante)
df_s2["cobertura_pct"] = 100 * df_s2["valida_km2"] / AOI_KM2
df_s2["agua_filtrada_km2"] = df_s2["agua_km2"].where(
    df_s2["cobertura_pct"] >= MIN_VALID_COVERAGE
)

n_ok = int(df_s2["agua_filtrada_km2"].notna().sum())
print(f"Meses con cobertura >= {MIN_VALID_COVERAGE} %: {n_ok} de {len(df_s2)}")
display(df_s2.tail(10))

plt.figure(figsize=(13, 4))
plt.plot(df_s2["date"], df_s2["agua_km2"], color="steelblue", lw=1, alpha=0.35,
         label="S2 todos los meses")
plt.plot(df_s2["date"], df_s2["agua_filtrada_km2"], color="darkorange", lw=1.5,
         marker="o", ms=3, label=f"S2 cobertura >= {MIN_VALID_COVERAGE} %")
plt.title(f"Agua superficial mensual Sentinel-2 MNDWI sobre el AOI ({S2_START} -> {S2_END})")
plt.ylabel("km2 con agua")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# COMPLEMENTO M8c: serie MENSUAL de área con agua (JRC MonthlyHistory)
# ------------------------------------------------------------
# Totalmente automático:
#   1. Consulta a GEE el rango NATIVO real del asset.
#   2. Lo intersecta con la ventana de CONFIG (FECHA_INICIO/FECHA_FIN).
#   3. Itera solo sobre los años de ANIOS_COMPLETOS dentro de esa intersección.
# No hay años escritos a mano. Si cambias CONFIG, esto se ajusta solo.
# ============================================================

ASSET = "JRC/GSW1_4/MonthlyHistory"

# --- 1) Rango nativo REAL del asset (consulta a GEE) -------------------------
_mh_todo = ee.ImageCollection(ASSET)
_t_ini_ms = _mh_todo.aggregate_min("system:time_start")
_t_fin_ms = _mh_todo.aggregate_max("system:time_start")
rango = ee.Dictionary({
    "ini": ee.Date(_t_ini_ms).format("YYYY-MM-dd"),
    "fin": ee.Date(_t_fin_ms).format("YYYY-MM-dd"),
}).getInfo()

nativo_ini = pd.Timestamp(rango["ini"])
nativo_fin = pd.Timestamp(rango["fin"])
print(f"{ASSET} nativo: {nativo_ini:%Y-%m} → {nativo_fin:%Y-%m}")

# --- 2) Intersección con CONFIG ---------------------------------------------
cfg_ini = pd.Timestamp(FECHA_INICIO)
cfg_fin = pd.Timestamp(FECHA_FIN)

ef_ini = max(cfg_ini, nativo_ini)
ef_fin = min(cfg_fin, nativo_fin)

if ef_ini > ef_fin:
    raise ValueError(
        f"Sin solapamiento entre CONFIG ({cfg_ini:%Y-%m}→{cfg_fin:%Y-%m}) "
        f"y el asset ({nativo_ini:%Y-%m}→{nativo_fin:%Y-%m})."
    )

INI_STR = ef_ini.strftime("%Y-%m-%d")
FIN_STR = ef_fin.strftime("%Y-%m-%d")
print(f"Ventana efectiva (CONFIG ∩ nativo): {INI_STR} → {FIN_STR}")

# --- 3) Años a procesar = ANIOS_COMPLETOS dentro de la ventana efectiva -----
ANIOS_DATASET = [y for y in ANIOS_COMPLETOS if ef_ini.year <= y <= ef_fin.year]
print(f"Años a procesar ({len(ANIOS_DATASET)}): {ANIOS_DATASET[0]} … {ANIOS_DATASET[-1]}")

# --- Colección filtrada -----------------------------------------------------
mh = ee.ImageCollection(ASSET).filterBounds(roi).filterDate(INI_STR, FIN_STR)
print(f"Imágenes en el AOI: {mh.size().getInfo()}")

try:
    pix_km2
except NameError:
    pix_km2 = ee.Image.pixelArea().divide(1e6)


def _area_mes_jrc(img):
    """Área (km²) con agua y con dato para una imagen mensual del asset."""
    water = img.select("water")          # 0 = sin dato, 1 = no agua, 2 = agua
    agua = water.eq(2)
    valida = water.gt(0)
    stacked = (pix_km2.updateMask(agua).rename("agua_km2")
               .addBands(pix_km2.updateMask(valida).rename("valida_km2")))
    st = stacked.reduceRegion(
        ee.Reducer.sum(), roi, ESCALAS["jrc"],
        maxPixels=1e10, tileScale=8, bestEffort=False,
    )
    return ee.Feature(None, {
        "anio": ee.Date(img.get("system:time_start")).get("year"),
        "mes":  ee.Date(img.get("system:time_start")).get("month"),
        "agua_km2":   st.get("agua_km2"),
        "valida_km2": st.get("valida_km2"),
    })


# --- Iteración por año (evita el getInfo() gigante) -------------------------
recs = []
for y in ANIOS_DATASET:
    sub = mh.filter(ee.Filter.calendarRange(y, y, "year"))
    n_anio = sub.size().getInfo()
    if n_anio == 0:
        print(f"  {y}: sin imágenes, se omite")
        continue
    try:
        fc = ee.FeatureCollection(sub.map(_area_mes_jrc)).getInfo()
        recs += [f["properties"] for f in fc["features"]]
        print(f"  {y}: {n_anio} meses")
    except Exception as e:
        print(f"  {y}: error — {e}")

# --- DataFrame --------------------------------------------------------------
df_mh = pd.DataFrame(recs)
df_mh["anio"] = pd.to_numeric(df_mh["anio"], errors="coerce").astype("Int64")
df_mh["mes"]  = pd.to_numeric(df_mh["mes"],  errors="coerce").astype("Int64")
df_mh[["agua_km2", "valida_km2"]] = df_mh[["agua_km2", "valida_km2"]].apply(
    pd.to_numeric, errors="coerce"
)
df_mh["date"] = pd.to_datetime(dict(year=df_mh["anio"], month=df_mh["mes"], day=1))
df_mh = df_mh.sort_values("date").reset_index(drop=True)

# Cobertura real (denominador = AOI completo, no los píxeles con dato)
df_mh["cobertura_pct"] = 100 * df_mh["valida_km2"] / AOI_KM2

f_ini, f_fin = df_mh["date"].min(), df_mh["date"].max()
print(f"\nSerie efectiva: {f_ini:%Y-%m} → {f_fin:%Y-%m}  ({len(df_mh)} meses)")
print(f"Cobertura ≥ 80 %: {(df_mh['cobertura_pct'] >= 80).sum()} meses")

df_mh_ok = df_mh[df_mh["cobertura_pct"] >= 80].copy()

# --- Gráfico ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(df_mh["date"], df_mh["agua_km2"],
        color="lightblue", lw=0.8, label="Todos los meses")
ax.plot(df_mh_ok["date"], df_mh_ok["agua_km2"],
        color="navy", lw=1.2, marker=".", ms=3,
        label="Cobertura ≥ 80 %")
ax.set_title(f"Área mensual con agua — JRC MonthlyHistory "
             f"({f_ini:%Y-%m} → {f_fin:%Y-%m})")
ax.set_ylabel("km² con agua")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# M8i: síntesis — precipitación vs área de agua
# ============================================================
precip_anual = (df_chirps.assign(anio=df_chirps["date"].dt.year)
                .groupby("anio", as_index=False)["mm"].sum())

# df_agua_anio viene de la celda M8.2 del notebook base
merged = df_agua_anio.merge(precip_anual, on="anio", how="inner")
corr = merged[["mm", "km2_agua"]].corr().iloc[0, 1]
print(f"Correlación precipitación ↔ área de agua: r = {corr:.2f}")
print(f"n años emparejados: {len(merged)}")

fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax1.bar(merged["anio"], merged["mm"], color="skyblue", alpha=0.6)
ax1.set_ylabel("Precipitación (mm/año)", color="skyblue")
ax2 = ax1.twinx()
ax2.plot(merged["anio"], merged["km2_agua"], color="darkorange",
         lw=2, marker="o", ms=4)
ax2.set_ylabel("km² con agua", color="darkorange")
plt.title(f"Precipitación anual vs área de agua superficial (r = {corr:.2f})")
plt.tight_layout(); plt.show()

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
